# Gold 3_18 — Coaching Score by Client Health

**This table grades the COMPANY, not the client.**

The health score (3_13/3_14) grades the student. This table inverts the gaze: one row per client, it measures whether the company kept the promises implied by a premium coaching contract — reply to your client, deliver the coaching you sold, initiate service. Pure context table: NOT an input to any score, changes no existing rule.

## The standard (provisional — business may retune)

* A paying client who reaches out gets a reply within 24 hours. **2 days is the last acceptable day.**
* 3+ days unanswered = the promise is slipping. 30+ days unanswered with no attempt = the promise is abandoned.
* A paying client with zero meetings ever recorded = a contract without delivery.
* No company-initiated contact in 30+ days on a paying client = minimal service.

## The promise ladder (`service_verdict`)

| Band | Meaning |
| --- | --- |
| PROMISE KEPT | no reply is owed — our move was made |
| WITHIN THE PROMISE (0-2 days) | the last acceptable day |
| PROMISE SLIPPING (3-7 days) | past the 2-day promise |
| PROMISE BROKEN (8-30 days) | the client is waiting on a broken promise |
| PROMISE ABANDONED (31+ days) | a month or more, and nobody tried |

## Column dictionary (names stay professional — the analysis voice lives in the column comments)

* `days_awaiting_reply` / `reply_days_past_promise` — the reply debt (NULL if none owed)
* `no_attempt_since_reach` — no outbound attempt of ANY kind after the client's last message
* `promise_overdue_3d` / `promise_overdue_30d` / `paying_and_unanswered_30d` — severity of the reply debt
* `company_quiet_days` / `company_minimal_service` — how long since WE initiated anything
* `no_meeting_recorded_ever` / `no_meeting_scheduled` / `contract_without_delivery` — the delivery record
* `company_service_score` — 0–100 penalty-based summary (PROVISIONAL weights; the verdict is never score-diluted)
* `health_band`, `final_health_score` — the company's own grade of the CLIENT, carried through for the side-by-side

Every column carries a COMMENT in the table metadata describing **how the client experiences that state** — the honest framing survives inside the data dictionary, where no one can call it editorial.

## How customers experience this — and why these numbers are business risk

Bad news travels fast. Customers who feel let down share their experience far more often — and far more publicly — than satisfied ones, and complaints reach more people faster than praise ever does. For a coaching company the exposure doubles: **the product IS the relationship**, so a service failure is a product failure a client can state in one sentence — "I paid them five figures and they went silent for a month."

The "silent" clients in this table are only silent in the CRM. PROMISE BROKEN and PROMISE ABANDONED rows are where public reviews, refund demands, and chargebacks come from. The reply debt is not just a service metric — it is the raw material of the company's public reputation, and the money sitting in `contract_without_delivery` accounts is a liability number, not just a revenue number.

## Known limitations (honest)

* `Contracted_Value` missing for ~half of clients — `paying_client` means "contract on file," not "didn't pay."
* `date_of_sale` present for only 171 of 1,630 — `days_since_sale` is context only.
* Meeting absence depends on the company's own logging — if meetings happen but aren't recorded, that is a data-governance failure, and it is the company's either way.

In [0]:
# ============================================================================
# BUILD - crm_ingestion.gold.coaching_score_by_client_health
# ============================================================================
# One row per client. Grades the COMPANY's service, not the client's engagement.
# Sources: gold.customer_health_profile (3_14, now SCD2 — OPEN versions joined for
# the company's own metrics, carried through for the side-by-side) +
# gold.communication_recency (3_17: the reply debt and contact/attempt record) +
# gold.health_metrics (csm_name / Tier / program / days_until_next_meeting —
# commercial context that is not part of the SCD2 version record).
# Thresholds are PROVISIONAL constants - business may retune without touching
# the pipeline. This table is NOT an input to any score.

spark.sql("""
CREATE OR REPLACE TABLE crm_ingestion.gold.coaching_score_by_client_health AS

WITH base AS (
    -- customer_health_profile is now SCD2: open versions (VRSN_END_DATE IS NULL)
    -- are the current clients. Column names carried through from the old schema
    -- (lead_name/health_band/final_health_score/calc_days_since_last_meeting/...)
    -- so the derived logic below is unchanged. csm_name / Tier / program /
    -- days_until_next_meeting come from health_metrics (not versioned).
    SELECT
        p.LEAD_ID                                       AS lead_id,
        p.LEAD_NAME                                     AS lead_name,
        h.csm_name,
        h.Tier,
        h.program,
        p.HEALTH_BAND                                   AS health_band,
        p.FINAL_SCORE                                   AS final_health_score,
        p.DAYS_SINCE_LAST_MEETING                       AS calc_days_since_last_meeting,
        h.days_until_next_meeting                       AS calc_days_until_next_meeting,
        p.CONTRACT_VALUE                                AS contract_value,
        p.CONTRACT_START_DATE                           AS sale_date,
        cr.whose_turn,
        cr.last_contact_in_days,
        cr.last_contact_out_days,
        cr.last_attempt_days
    FROM crm_ingestion.gold.customer_health_profile p
    JOIN crm_ingestion.gold.communication_recency cr ON cr.lead_id = p.LEAD_ID
    LEFT JOIN crm_ingestion.gold.health_metrics h     ON h.lead_id = p.LEAD_ID
    WHERE p.VRSN_END_DATE IS NULL
),

derived AS (
    SELECT
        b.*,
        (b.contract_value IS NOT NULL AND b.contract_value > 0)            AS paying_client,
        DATEDIFF(current_date, b.sale_date)                               AS days_since_sale,
        CASE WHEN b.whose_turn = 'OUR TURN' THEN b.last_contact_in_days END AS days_awaiting_reply,
        CASE WHEN b.whose_turn = 'OUR TURN' AND b.last_contact_in_days > 2
             THEN b.last_contact_in_days - 2 END                           AS reply_days_past_promise,
        (b.whose_turn = 'OUR TURN')                                        AS reply_owed,
        CASE WHEN b.whose_turn = 'OUR TURN'
              AND (b.last_attempt_days IS NULL OR b.last_attempt_days >= b.last_contact_in_days)
             THEN 1 ELSE 0 END                                             AS no_attempt_since_reach,
        CASE WHEN b.whose_turn = 'OUR TURN' AND b.last_contact_in_days >= 3
             THEN 1 ELSE 0 END                                             AS promise_overdue_3d,
        CASE WHEN b.whose_turn = 'OUR TURN' AND b.last_contact_in_days > 30
              AND (b.last_attempt_days IS NULL OR b.last_attempt_days >= b.last_contact_in_days)
             THEN 1 ELSE 0 END                                             AS promise_overdue_30d,
        (b.calc_days_since_last_meeting IS NULL)                           AS no_meeting_recorded_ever,
        (b.calc_days_until_next_meeting IS NULL)                           AS no_meeting_scheduled,
        b.last_contact_out_days                                           AS company_quiet_days,
        CASE WHEN b.last_contact_out_days IS NULL OR b.last_contact_out_days > 30
             THEN 1 ELSE 0 END                                             AS company_minimal_service
    FROM base b
),

flags AS (
    SELECT
        d.*,
        CASE WHEN d.paying_client AND d.promise_overdue_30d = 1 THEN 1 ELSE 0 END AS paying_and_unanswered_30d,
        CASE WHEN d.paying_client AND d.no_meeting_recorded_ever  THEN 1 ELSE 0 END AS contract_without_delivery
    FROM derived d
),

scored AS (
    SELECT
        f.*,
        GREATEST(0, 100 - (
              LEAST(40, 10 * COALESCE(f.reply_days_past_promise, 0))
            + CASE WHEN f.no_attempt_since_reach    = 1 THEN 20 ELSE 0 END
            + CASE WHEN f.promise_overdue_30d       = 1 THEN 40 ELSE 0 END
            + CASE WHEN f.contract_without_delivery = 1 THEN 30 ELSE 0 END
            + CASE WHEN f.company_minimal_service   = 1 THEN 10 ELSE 0 END
        )) AS company_service_score
    FROM flags f
)

SELECT
    lead_id, lead_name,
    csm_name, Tier, program,
    contract_value, paying_client, days_since_sale,
    health_band, final_health_score,
    days_awaiting_reply, reply_days_past_promise,
    reply_owed, no_attempt_since_reach,
    promise_overdue_3d, promise_overdue_30d,
    company_quiet_days, company_minimal_service,
    no_meeting_recorded_ever, no_meeting_scheduled,
    contract_without_delivery,
    paying_and_unanswered_30d,
    company_service_score,
    CASE
        WHEN NOT reply_owed           THEN 'PROMISE KEPT'
        WHEN days_awaiting_reply <= 2 THEN 'WITHIN THE PROMISE (0-2 days)'
        WHEN days_awaiting_reply <= 7 THEN 'PROMISE SLIPPING (3-7 days)'
        WHEN days_awaiting_reply <= 30 THEN 'PROMISE BROKEN (8-30 days)'
        ELSE 'PROMISE ABANDONED (31+ days)'
    END AS service_verdict
FROM scored
""")

# ------------------------------------------------------------------------
# Column comments: how the CLIENT experiences each state. The analysis
# voice lives in the data dictionary - visible to anyone who queries the
# table, impossible to dismiss as a column-name stunt.
# ------------------------------------------------------------------------
column_comments = {}
column_comments["days_awaiting_reply"] = "Days the client has waited for our reply. Every day here is a day the client feels deprioritized."
column_comments["reply_days_past_promise"] = "Days beyond the 2-day promise window for a reply."
column_comments["reply_owed"] = "True when a client message is waiting on us. The client knows exactly how long it has been."
column_comments["no_attempt_since_reach"] = "No outbound attempt of any kind since the client last reached out. The client does not know we are busy - they only know we are silent."
column_comments["promise_overdue_3d"] = "Client waiting 3+ days for a reply. Past this point the client experiences the promise as broken."
column_comments["promise_overdue_30d"] = "Client waiting 30+ days with no attempt to reach them. From the client side, this is abandonment."
column_comments["paying_and_unanswered_30d"] = "A PAYING client has been waiting 30+ days. Highest churn, refund, and public-complaint risk on the book."
column_comments["company_quiet_days"] = "Days since we initiated any contact. Silence is not experienced as service."
column_comments["company_minimal_service"] = "No company-initiated contact in 30+ days on a paying client."
column_comments["no_meeting_recorded_ever"] = "No meeting has ever been recorded for this client, by the company metric itself."
column_comments["no_meeting_scheduled"] = "No next meeting on the books. The client has nothing scheduled to look forward to."
column_comments["contract_without_delivery"] = "Paying client with zero recorded meetings ever. What was sold and what was delivered do not match."
column_comments["company_service_score"] = "0-100 penalty-based summary of company service failures. Provisional weights; the verdict band is never score-diluted."
column_comments["service_verdict"] = "The promise ladder: KEPT, WITHIN THE PROMISE, SLIPPING, BROKEN, ABANDONED."
column_comments["health_band"] = "The company grade of the CLIENT - shown deliberately beside the company grade of itself."

for col in column_comments:
    sql_stmt = (
        "ALTER TABLE crm_ingestion.gold.coaching_score_by_client_health "
        + "ALTER COLUMN " + col + " COMMENT '" + column_comments[col] + "'"
    )
    spark.sql(sql_stmt)

table_comment_stmt = (
    "COMMENT ON TABLE crm_ingestion.gold.coaching_score_by_client_health IS "
    + "'Company service scorecard, one row per client - measures promise-keeping: "
    + "reply SLA, delivery, initiation. Context table, NOT an input to any score. "
    + "Thresholds provisional.'"
)
spark.sql(table_comment_stmt)

result = spark.table("crm_ingestion.gold.coaching_score_by_client_health")
print("coaching_score_by_client_health built: {:,} clients".format(result.count()))
result.groupBy("service_verdict").count().orderBy("count", ascending=False).show()

In [0]:
%sql
-- Validation 1: the promise ladder — the company's report card
SELECT
    CASE WHEN service_verdict LIKE 'PROMISE KEPT%'    THEN 0
         WHEN service_verdict LIKE 'WITHIN THE%'      THEN 1
         WHEN service_verdict LIKE 'PROMISE SLIPPING%' THEN 2
         WHEN service_verdict LIKE 'PROMISE BROKEN%'  THEN 3
         ELSE 4 END AS ord,
    service_verdict,
    COUNT(*) AS clients,
    ROUND(AVG(company_service_score), 0) AS avg_score,
    SUM(paying_and_unanswered_30d) AS paying_and_unanswered_30d,
    SUM(contract_without_delivery) AS contract_without_delivery,
    SUM(promise_overdue_3d)        AS promise_overdue_3d
FROM crm_ingestion.gold.coaching_score_by_client_health
GROUP BY service_verdict
ORDER BY ord

In [0]:
%sql
-- Validation 2: THE MONEY — contract dollars sitting in failed-service states.
-- The liability view: not revenue at risk of churn, but payment already taken
-- against delivery that has not happened.
SELECT
    COUNT(*)                                                          AS paying_clients,
    ROUND(SUM(contract_value), 0)                                     AS total_contract_value,
    ROUND(SUM(CASE WHEN contract_without_delivery  = 1 THEN contract_value END), 0) AS dollars_contract_without_delivery,
    ROUND(SUM(CASE WHEN paying_and_unanswered_30d = 1 THEN contract_value END), 0) AS dollars_paying_unanswered_30d,
    ROUND(SUM(CASE WHEN company_minimal_service    = 1 THEN contract_value END), 0) AS dollars_on_minimal_service
FROM crm_ingestion.gold.coaching_score_by_client_health
WHERE paying_client

In [0]:
%sql
-- Validation 3: the side-by-side — the company's grade of the CLIENT vs the
-- company's own service verdict. Where clients are graded POOR/NO DATA while
-- the company owes them a reply, the health score is grading the victim.
SELECT
    service_verdict,
    health_band,
    COUNT(*) AS clients
FROM crm_ingestion.gold.coaching_score_by_client_health
GROUP BY service_verdict, health_band
ORDER BY
    CASE WHEN service_verdict LIKE 'PROMISE KEPT%'    THEN 0
         WHEN service_verdict LIKE 'WITHIN THE%'      THEN 1
         WHEN service_verdict LIKE 'PROMISE SLIPPING%' THEN 2
         WHEN service_verdict LIKE 'PROMISE BROKEN%'  THEN 3
         ELSE 4 END,
    health_band

In [0]:
%sql
-- Validation 4: the 15 accounts needing immediate attention — lowest service
-- scores first, money breaking ties. Named CSMs: accountability only works if
-- it points at owners.
SELECT
    lead_name,
    COALESCE(csm_name, '-') AS CSM,
    Tier,
    contract_value,
    days_awaiting_reply,
    no_attempt_since_reach,
    company_quiet_days,
    no_meeting_recorded_ever,
    health_band,
    company_service_score,
    service_verdict
FROM crm_ingestion.gold.coaching_score_by_client_health
ORDER BY company_service_score ASC, contract_value DESC
LIMIT 15

In [0]:
%sql
-- Validation 5: CSM accountability rollup — the whole book per CSM.
-- Total clients, failure counts, failure RATES (rates over raw counts: a big
-- book fails more in absolute terms by volume alone), and dollars at risk.
-- This answers: is the service failure company-wide, or a few bad books?
SELECT
    COALESCE(NULLIF(csm_name, ''), '(unassigned)') AS csm,
    COUNT(*) AS total_clients,
    SUM(CAST(paying_client AS INT)) AS paying_clients,
    ROUND(SUM(contract_value), 0) AS contract_value,
    ROUND(AVG(company_service_score), 1) AS avg_service_score,
    SUM(promise_overdue_3d) AS clients_past_promise,
    ROUND(100.0 * SUM(promise_overdue_3d) / COUNT(*), 1) AS past_promise_pct,
    SUM(promise_overdue_30d) AS abandoned_30d,
    SUM(no_attempt_since_reach) AS zero_attempts_after_reach,
    SUM(contract_without_delivery) AS never_delivered,
    SUM(company_minimal_service) AS minimal_service,
    ROUND(SUM(CASE WHEN contract_without_delivery = 1 THEN contract_value END), 0) AS dollars_never_delivered
FROM crm_ingestion.gold.coaching_score_by_client_health
GROUP BY COALESCE(NULLIF(csm_name, ''), '(unassigned)')
ORDER BY clients_past_promise DESC, total_clients DESC

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Validation 6: COMPANY-WIDE OR BAD APPLES — the concentration test.
-- Company baseline failing rate, each CSM vs that baseline, and each CSM's
-- share of ALL past-promise accounts. If 2-3 CSMs hold most of the failures
-- while most CSMs sit at zero -> individuals. If failures spread across most
-- of the bench -> systemic. Book size shown for workload context: an
-- overloaded CSM is a staffing problem, not a character problem.
WITH csm AS (
    SELECT
        COALESCE(NULLIF(csm_name, ''), '(unassigned)') AS csm,
        COUNT(*) AS total_clients,
        SUM(promise_overdue_3d) AS past_promise
    FROM crm_ingestion.gold.coaching_score_by_client_health
    GROUP BY 1
),
company AS (
    SELECT SUM(total_clients) AS all_clients, SUM(past_promise) AS all_past_promise FROM csm
)
SELECT
    csm.csm,
    csm.total_clients,
    csm.past_promise,
    ROUND(100.0 * csm.past_promise / csm.total_clients, 1) AS csm_failing_pct,
    ROUND(100.0 * company.all_past_promise / company.all_clients, 1) AS company_failing_pct,
    ROUND(100.0 * csm.past_promise / company.all_past_promise, 1) AS share_of_all_failures,
    ROUND(100.0 * csm.total_clients / company.all_clients, 1) AS share_of_all_clients
FROM csm, company
ORDER BY share_of_all_failures DESC, total_clients DESC

In [0]:
%sql
-- Validation 7: the promise ladder BY CSM — each book's distribution across
-- the ladder. The split between SLIPPING (fresh debt, this week) and
-- BROKEN/ABANDONED (old debt, institutional) tells you whether a book is
-- running behind or has been behind for a long time.
SELECT
    COALESCE(NULLIF(csm_name, ''), '(unassigned)') AS csm,
    COUNT(*) AS total_clients,
    SUM(CASE WHEN service_verdict LIKE 'PROMISE KEPT%'     THEN 1 ELSE 0 END) AS kept,
    SUM(CASE WHEN service_verdict LIKE 'WITHIN THE%'      THEN 1 ELSE 0 END) AS within_promise,
    SUM(CASE WHEN service_verdict LIKE 'PROMISE SLIPPING%' THEN 1 ELSE 0 END) AS slipping,
    SUM(CASE WHEN service_verdict LIKE 'PROMISE BROKEN%'  THEN 1 ELSE 0 END) AS broken,
    SUM(CASE WHEN service_verdict LIKE 'PROMISE ABANDONED%' THEN 1 ELSE 0 END) AS abandoned,
    ROUND(AVG(company_service_score), 1) AS avg_score
FROM crm_ingestion.gold.coaching_score_by_client_health
GROUP BY 1
ORDER BY slipping + broken + abandoned DESC, total_clients DESC

In [0]:
%sql
-- Chart data: the promise ladder by CSM in LONG format (one row per CSM x
-- band) so the stacked chart can color by band. Wide-format lives in
-- Validation 7; this cell exists to feed the visualization.
SELECT
    COALESCE(NULLIF(csm_name, ''), '(unassigned)') AS csm,
    service_verdict AS promise_band,
    COUNT(*) AS clients
FROM crm_ingestion.gold.coaching_score_by_client_health
GROUP BY 1, 2
ORDER BY csm,
    CASE WHEN service_verdict LIKE 'PROMISE KEPT%'     THEN 0
         WHEN service_verdict LIKE 'WITHIN THE%'      THEN 1
         WHEN service_verdict LIKE 'PROMISE SLIPPING%' THEN 2
         WHEN service_verdict LIKE 'PROMISE BROKEN%'  THEN 3
         ELSE 4 END

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Validation 8: the economics of the bench — could the company afford to
-- answer its clients? Assumes $200K fully-loaded cost per CSM per year
-- (working assumption; swap the constant and rerun). Contracts are 12 months
-- or less, so the current book IS the annual book.
-- NOTE: contract values exist for only ~half of clients — this total is a
-- FLOOR; the real book is larger.
WITH book AS (
    SELECT
        SUM(CAST(paying_client AS INT)) AS paying_clients,
        ROUND(SUM(contract_value), 0) AS total_contract
    FROM crm_ingestion.gold.coaching_score_by_client_health
    WHERE paying_client
),
bench AS (
    SELECT COUNT(DISTINCT csm_name) AS named_csms
    FROM crm_ingestion.gold.coaching_score_by_client_health
    WHERE COALESCE(csm_name, '') <> ''
)
SELECT
    b.paying_clients,
    b.total_contract,
    ROUND(b.total_contract / b.paying_clients, 0) AS avg_contract_per_client,
    n.named_csms,
    200000 AS assumed_csm_cost,
    ROUND(n.named_csms * 200000, 0) AS bench_payroll,
    ROUND(100.0 * n.named_csms * 200000 / b.total_contract, 1) AS payroll_pct_of_book,
    ROUND(b.total_contract - n.named_csms * 200000, 0) AS left_after_payroll,
    ROUND(b.total_contract / n.named_csms, 0) AS book_per_csm,
    ROUND(b.total_contract / n.named_csms / 200000, 1) AS salaries_covered_per_csm
FROM book b, bench n

## Closing note — analyst's disclosure

I built this table as a client of the company it measures.

The service standard applied here — a paying client gets a reply within 24 hours; two days is the last acceptable day — was written from the client's side of the mailbox, then tested against the company's own CRM record for all 1,630 clients. The table, the CSM rollup, and the concentration test were complete before I looked up where I sit in this data.

My CSM holds the worst-performing book on the bench by every measure this notebook computes: the highest failure rate (16.0% against a 6.0% company baseline), the only book carrying debt at every rung of the promise ladder, and 34% of the company's service failures — while carrying fewer clients than three peers who fail less. My first CSM appears in this data as the smallest book in the company: one client, nothing delivered on record. I was reassigned from that book to this one, and the difference was immediate.

The analysis was not built to find her. It was built blind — and it landed on the book I already lived in. The experience that motivated this work and the data that documents it agree.

For anyone reading this as a hiring manager or reviewer: this is the point of the project. The analyst had a hypothesis formed as a customer, built an instrument to test it without knowing where he would appear in the results, and the instrument confirmed the hypothesis at company scale — including on the account he was standing in. Every number in this notebook is reproducible from the company's own CRM export.

In [0]:
# ============================================================================
# CHANGE-DATA CONTRACTS — PKs, CHECKs, NOT NULL, table comments
# Rewritten 2026-11-13 (the Nov-12 cells were lost to a bad edit and never
# persisted — catalog verified empty before this rewrite).
# ============================================================================
# DESIGN FACTS BAKED IN (pre-flighted against live data 2026-11-13):
#   * PRIMARY KEY columns must be SET NOT NULL first — every table here was
#     built by CTAS, which declares all columns nullable (that is what killed
#     the Nov-12 attempt at its very first statement).
#   * SCD2 profile PK = (LEAD_ID, VRSN_START_DATE, INSERT_DATE). The originally
#     proposed (LEAD_ID, VRSN_START_DATE) is NOT unique — 380 same-day
#     close/reopen pairs exist from multi-run days; INSERT_DATE disambiguates
#     the run that opened the version (all three stronger keys verified 0-dup).
#   * No delta.appendOnly / immutability properties: customer_health_history
#     deletes same-day rows for idempotent re-runs and prunes >90 days BY
#     DESIGN — append-only would break the pipeline.
#   * One-open-version-per-lead is a table-level invariant (not expressible
#     as a column CHECK) — re-verified on every run of this cell.
#   * WIRED INTO THE NIGHTLY RUN (2026-11-13): this cell is the last cell of
#     3_18, the TERMINAL task of the DAG — it runs after every nightly CTAS
#     rebuild (health_metrics, communication_state, communication_recency,
#     coaching_score_by_client_health) and re-applies what those rebuilds
#     drop. Only MANUAL rebuilds outside the job (e.g. a silver full rebuild)
#     need this cell re-run by hand. Idempotent: existing constraints are
#     skipped.
#   * Silver id contracts: APPLIED since the 2026-11-13 cleanup (90 legacy
#     NULL-id rows deleted — failed schema parses + source-side empty records
#     — and the 2 repair-mangled activities backfilled). The NULL-id guard
#     stays live as a tripwire: it BLOCKS loudly rather than applying a PK
#     over a bad parse, which is how the repair-UDF mangling was caught.

G = "crm_ingestion.gold."
S = "crm_ingestion.silver."

_applied, _skipped, _failed, _blocked = 0, 0, 0, 0

def _one(sql):
    return spark.sql(sql).collect()[0][0]

def _ddl(label, sql):
    global _applied, _skipped, _failed
    try:
        spark.sql(sql)
        _applied += 1
        print(f"APPLIED | {label}")
    except Exception as e:
        m = str(e)
        if "already exists" in m.lower():
            _skipped += 1
            print(f"SKIP    | {label} (already present)")
        else:
            _failed += 1
            print(f"FAIL    | {label} :: {m[:180]}")

def _dup_count(table, cols):
    c = cols if isinstance(cols, str) else ", ".join(cols)
    return (f"SELECT COUNT(*) FROM (SELECT {c} FROM {table} "
            f"GROUP BY {c} HAVING COUNT(*) > 1)")

def apply_group(label, guard_sql, guard_desc, ddls):
    """Apply DDL only if the guard query returns 0 (no violating rows)."""
    global _blocked
    if guard_sql is not None:
        v = _one(guard_sql)
        if v != 0:
            _blocked += 1
            print(f"BLOCKED | {label} :: {guard_desc} = {v}")
            return
    for l, sql in ddls:
        _ddl(l, sql)

print("=" * 78)
print("CHANGE-DATA CONTRACTS — 2026-11-13")
print("=" * 78)

# -- 1. SCD2 profile: the version record --------------------------------------
P = G + "customer_health_profile"
print("\n[1] customer_health_profile (SCD2)")
apply_group("profile NOT NULL flips",
    f"SELECT COUNT(*) FROM {P} WHERE LEAD_ID IS NULL OR VRSN_START_DATE IS NULL OR INSERT_DATE IS NULL",
    "key NULLs",
    [("profile.LEAD_ID SET NOT NULL",         f"ALTER TABLE {P} ALTER COLUMN LEAD_ID SET NOT NULL"),
     ("profile.VRSN_START_DATE SET NOT NULL", f"ALTER TABLE {P} ALTER COLUMN VRSN_START_DATE SET NOT NULL"),
     ("profile.INSERT_DATE SET NOT NULL",     f"ALTER TABLE {P} ALTER COLUMN INSERT_DATE SET NOT NULL")])
apply_group("profile PK (LEAD_ID, VRSN_START_DATE, INSERT_DATE)",
    _dup_count(P, "LEAD_ID, VRSN_START_DATE, INSERT_DATE"),
    "dup version-key triples",
    [("profile PK", f"ALTER TABLE {P} ADD CONSTRAINT pk_customer_health_profile PRIMARY KEY (LEAD_ID, VRSN_START_DATE, INSERT_DATE)")])
apply_group("profile CHECK vrsn order",
    f"SELECT COUNT(*) FROM {P} WHERE VRSN_END_DATE IS NOT NULL AND VRSN_END_DATE < VRSN_START_DATE",
    "end-before-start violations",
    [("profile CHECK (closed >= open)",
      f"ALTER TABLE {P} ADD CONSTRAINT chk_customer_health_profile_vrsn_order CHECK (VRSN_END_DATE IS NULL OR VRSN_END_DATE >= VRSN_START_DATE)")])
_mo = _one(f"SELECT COUNT(*) FROM (SELECT LEAD_ID FROM {P} WHERE VRSN_END_DATE IS NULL GROUP BY LEAD_ID HAVING COUNT(*) > 1)")
print(f"{'OK     ' if _mo == 0 else 'WARN!  '}| profile invariant: leads with >1 open version = {_mo} (table-level, not a CHECK)")

# -- 2. Daily history: the snapshot grain ------------------------------------
H = G + "customer_health_history"
print("\n[2] customer_health_history (daily snapshot)")
apply_group("history NOT NULL+PK",
    _dup_count(H, "run_date, LEAD_ID"), "NULL/dup (run_date, LEAD_ID)",
    [("history.run_date SET NOT NULL", f"ALTER TABLE {H} ALTER COLUMN run_date SET NOT NULL"),
     ("history.LEAD_ID SET NOT NULL",  f"ALTER TABLE {H} ALTER COLUMN LEAD_ID SET NOT NULL"),
     ("history PK", f"ALTER TABLE {H} ADD CONSTRAINT pk_customer_health_history PRIMARY KEY (run_date, LEAD_ID)")])
apply_group("history CHECK open-only",
    f"SELECT COUNT(*) FROM {H} WHERE VRSN_END_DATE IS NOT NULL", "non-open rows",
    [("history CHECK (VRSN_END_DATE IS NULL)",
      f"ALTER TABLE {H} ADD CONSTRAINT chk_customer_health_history_open_only CHECK (VRSN_END_DATE IS NULL)")])

# -- 3. Daily-recomputed gold tables: one row per lead ------------------------
print("\n[3] lead_id contracts (health_metrics, communication_state, communication_recency, coaching_score)")
for tbl in ["health_metrics", "communication_state", "communication_recency", "coaching_score_by_client_health"]:
    T = G + tbl
    apply_group(f"{tbl} NOT NULL+PK", _dup_count(T, "lead_id"), "NULL/dup lead_id",
        [(f"{tbl}.lead_id SET NOT NULL", f"ALTER TABLE {T} ALTER COLUMN lead_id SET NOT NULL"),
         (f"{tbl} PK", f"ALTER TABLE {T} ADD CONSTRAINT pk_{tbl} PRIMARY KEY (lead_id)")])

# -- 4. Silver parents: id contracts (gated on NULL-id rows) ------------------
print("\n[4] silver id contracts (auto-skip while NULL-id rows exist)")
for tbl in ["lead_activites_processed", "close_crm_users_processed"]:
    T = S + tbl
    apply_group(f"silver.{tbl} NOT NULL+PK",
        f"SELECT (SELECT COUNT(*) FROM {T} WHERE id IS NULL) + ({_dup_count(T, 'id')}) AS v",
        "NULL/dup id",
        [(f"silver.{tbl}.id SET NOT NULL", f"ALTER TABLE {T} ALTER COLUMN id SET NOT NULL"),
         (f"silver.{tbl} PK", f"ALTER TABLE {T} ADD CONSTRAINT pk_{tbl} PRIMARY KEY (id)")])

# -- 5. Table comments (change-flow documentation, ungated) -------------------
print("\n[5] table comments")
COMMENTS = [
    (P,
     "SCD2 engagement version history: one row per lead per engagement version. "
     "Version grain key (LEAD_ID, VRSN_START_DATE, INSERT_DATE) — a same-day close/reopen "
     "makes two versions share a start date; INSERT_DATE disambiguates the run. A version "
     "closes when the engagement hash changes; VRSN_END_DATE IS NULL marks the current "
     "open version; exactly one open version per lead (verified invariant). Clock metrics "
     "are not hashed and refresh on the open version. NO DATA rows carry no scores by "
     "design. Built by 3_14; PK/CHECK via this contracts cell."),
    (H,
     "Rolling 90-day daily snapshot of OPEN customer_health_profile versions: one row per "
     "active lead per run date (run_date, LEAD_ID). Deliberately NOT append-only — same-day "
     "rows are deleted for idempotent re-runs and >90-day snapshots are pruned. Trend log "
     "for band transitions and score drift; what changed and when lives in the SCD2 profile. "
     "Built by 3_15."),
    (G + "health_metrics",
     "One row per Coaching Client lead — the engagement-metric assembly feeding the health "
     "score (3_14) and downstream context tables. Intern clients are INCLUDED and flagged "
     "is_intern_client (kept for tracking per the 3_13 contract). Rebuilt nightly by CTAS, "
     "which drops constraints — re-run the contracts cell after rebuilds. Built by 3_13."),
    (G + "communication_state",
     "Direction-aware per-channel communication recency for every lead seen in a "
     "communication source (all leads, not just clients). Attempt vs contact-made: an "
     "outbound call is an attempt unless answered; a client inbound call, even no-answer, "
     "means the client reached out. Population is signal-gated (guards against the all-leads "
     "sentiment spine flooding it). CONTEXT table — not an input to any health score. "
     "Rebuilt nightly by CTAS — re-run the contracts cell after rebuilds. Built by 3_16."),
    (G + "communication_recency",
     "Direction-aware communication recency and whose_turn verdict per coaching client. "
     "Attempts never flip the turn; contact-made does. CONTEXT ONLY — not an input to the "
     "health score (firewall by design). Rebuilt nightly by CTAS — re-run the contracts cell "
     "after rebuilds. Built by 3_17."),
    (G + "coaching_score_by_client_health",
     "Company service scorecard, one row per coaching client: grades whether the COMPANY "
     "kept its promises (reply debt, delivery, initiated service), not the client engagement. "
     "Weights and thresholds are PROVISIONAL and may be retuned without touching the "
     "pipeline. Not an input to any score. Rebuilt nightly by CTAS — re-run the contracts "
     "cell after rebuilds. Built by 3_18."),
    (S + "lead_activites_processed",
     "Silver parent for Close CRM activities (CDC by id). Rows with NULL id are failed schema "
     "parses of bronze chunks (see the 2026-11-13 diagnosis in the project record): unjoinable, "
     "excluded from every downstream join, and blocking the id contracts until removed."),
    (S + "close_crm_users_processed",
     "Silver parent for Close CRM users (CDC by id). Rows with NULL id are source-side empty "
     "records (first seen in the 2026-10-14 dump, plus later one-off empties): unjoinable, "
     "and blocking the id contracts until removed."),
]
for t, c in COMMENTS:
    _ddl(f"comment {t.split('.')[-1]}", f"COMMENT ON TABLE {t} IS '{c}'")

print("\n" + "=" * 78)
print(f"CONTRACTS SUMMARY: applied={_applied} skipped(already present)={_skipped} failed={_failed} blocked(guard)={_blocked}")
print("NOTE: CTAS rebuilds drop constraints on health_metrics / communication_state /")
print("communication_recency / coaching_score — re-run this cell after any rebuild.")
print("=" * 78)